# Class 5b - Survey Data (social science)

Recommended documentation: [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html), [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html), [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html), [statsmodels](https://www.statsmodels.org/stable/index.html)

In Class 4 and 5a, each row was a **time step** or a **grid cell** measured by an instrument. Today, each row is a **person** who answered a questionnaire. Survey data are the everyday material of social scientists working on forests: how owners see their forest, what they worry about, who decides what happens to it.

Goals:
- read survey data together with its **codebook**, and clean **missing-value codes**,
- analyse the main question types: single choice, multiple choice, **Likert scales** and **open text**,
- build **composite scores**, check their reliability, and find structure with **factor analysis**,
- compare groups with **statistical tests** and **regression**,
- use **machine learning** (scikit-learn) to explore and classify free-text answers.

> ### About this dataset (please read)
>
> **What:** the survey *"Wald haben – Die Bedeutung des Waldes für Besitzende und ihr Umfeld"* ("Owning a forest – what the forest means to owners and the people around them") was run in spring 2025 by the [DIANA project](https://diana-waldstrategie.de/) at the University of Freiburg. We use the answers of **501 private forest owners**, most of them owning small forests (under 20 ha) in Baden-Württemberg.
>
> **What we changed:** the file you get is a **modified teaching version**, not the original data:
> - only the questions we use today are kept (111 of 271 columns); timestamps, durations and most "other, please specify" answers are removed,
> - the codebook (questions and answer labels) is translated to English; open answers stay in their original language (German),
> - every respondent has a new random id and the rows are shuffled,
> - very rare answer categories (fewer than 5 people) are merged with a neighbouring category,
> - groups of answers (e.g. all attitude questions, or all contact questions) were **swapped between similar respondents** (same inheritance status, professional background and age group), so **no row corresponds to a real person** any more,
> - small random noise was added: about 5% of the scale answers moved by one step, and forest sizes were changed by a few percent,
> - names of people and places were removed from the free text.
>
> **Why:** the participants agreed that the DIANA team uses their answers for research, **not** that their answers are passed on. Even without names, a combination such as *age class + place of residence + forest size + a personal story in an open answer* can be enough to recognise someone in a small rural community. This is personal data under the GDPR (DSGVO) and must be protected.
>
> **Consequence:** the overall patterns are the same as in the real survey (we checked), so everything you learn today is valid. But the exact numbers are **not** the real results: do not cite them, share the files, or use them for research. If you need the real data, contact the DIANA team.

## 1. From sensor tables to survey tables

A survey table looks like any `DataFrame`, but the values are **codes** for answers. To read them you need the **codebook**: the list of questions, with the meaning of every code.

Question types you will meet today:

| Type | Example | Stored as |
|---|---|---|
| Single choice | age class, gender | one column, one code per person |
| Multiple choice | "How did you get your forest? (inherited / bought / ...)" | one **0/1 column per option** |
| Likert scale | "I feel emotionally connected to my forest", from *strongly agree* to *strongly disagree* | one column per statement, codes 1–5 |
| Intensity scale | "How strongly do you worry about drought?" | codes 1–5, plus 6 = *does not apply* |
| Free keywords | "Write up to 5 words that come to mind about your forest" | 5 text columns |
| Open question | "What would you do after a disaster in your forest?" | free text |

One more thing before we start: **this sample is not representative** of all German forest owners. People were recruited through forestry organisations, but also sports clubs, parishes, choirs... Keep that in mind whenever you generalise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Loading the data and the codebook

We read two CSV files with `pd.read_csv()` (Class 4): the answers, and the codebook.

In [ ]:
Owners = pd.read_csv("../processed_data/diana_owners.csv").copy()  # .copy() avoids warnings when adding columns
Codebook = pd.read_csv("../processed_data/diana_codebook.csv")

print(Owners.shape)
print(Owners.columns[:15])
print(Codebook.shape)

The column names are **question codes** (`F26`, `F10_v`, ...), as in real survey exports. The codebook tells us what they mean. Let's look up the question `F26`:

In [ ]:
print(Codebook[Codebook["variable"] == "F26"].T)

The `answer_labels` column is one long string: `"0=no answer; 3=18-34 years; ..."`. A small function turns it into a Python dictionary `{code: label}` that we can reuse for every question:

In [ ]:
def get_labels(Variable):
    # Turn "0=no answer; 1=male; ..." into {0: "no answer", 1: "male", ...}
    Text = Codebook.loc[Codebook["variable"] == Variable, "answer_labels"].iloc[0]
    Labels = {}
    for Pair in Text.split("; "):
        Code, Label = Pair.split("=", 1)
        Labels[int(Code)] = Label
    return Labels


def get_question(Variable):
    return Codebook.loc[Codebook["variable"] == Variable, "question"].iloc[0]


print(get_question("F26"))
print(get_labels("F26"))

### i. Missing-value codes

Survey software rarely writes empty cells. Instead, it uses **special codes**. In this dataset:

- `0` = *no answer* in all numeric questions,
- `6` = *does not apply* in the intensity scales (e.g. worries),
- `-99` = *not answered* and `-66` = *not shown* in text questions.

These codes are dangerous: if you compute the mean age class with the `0`s inside, the result is wrong. Always start with `value_counts(dropna=False)`, which also counts real `NaN`s:

In [ ]:
print(Owners["F26"].value_counts(dropna=False).sort_index())

### ii. From codes to labels

The usual cleaning steps are:

1. `.replace({0: np.nan})` turns the *no answer* code into a real missing value, so pandas ignores it in calculations,
2. `.map(Labels)` turns the codes into readable labels,
3. `pd.Categorical(..., ordered=True)` tells pandas that the labels have a **natural order** (18-34 < 35-44 < ...). Without it, plots and tables sort them alphabetically.

In [ ]:
AgeLabels = get_labels("F26")
AgeOrder = [AgeLabels[Code] for Code in sorted(AgeLabels) if Code != 0]

Age = Owners["F26"].replace({0: np.nan}).map(AgeLabels)
Age = pd.Categorical(Age, categories=AgeOrder, ordered=True)
Age = pd.Series(Age)

# normalize=True gives shares instead of counts; sort_index() keeps the category order
AgeShare = Age.value_counts(normalize=True).sort_index() * 100
print(AgeShare.round(1))

plt.bar(AgeShare.index, AgeShare)
plt.ylabel("Share of respondents (%)")
plt.title("Age of the participants")
plt.xticks(rotation=45)
plt.show()

## Exercise A

Almost every survey paper starts with a "**Table 1**" describing who answered. Reviewers use it to judge how far the results can be generalised.

Tasks:
1. Build a table with the share (%) of participants in each category of **age** (`F26`), **gender** (`F27`), **education** (`F28`) and **place of residence** (`F29`). Use `get_labels()` and do not count *no answer*.
2. Make one bar chart for each of **gender**, **education** and **residence**, as we did for age (tip: for long labels, `plt.barh` is easier to read).
3. Compare with Germany as a whole (about 50% women, about 30% of people living in rural areas). Who is over- or under-represented in this sample? What does it mean for our conclusions?

In [ ]:
# Solution
Table1 = []
for Variable in ["F26", "F27", "F28", "F29"]:
    Labels = get_labels(Variable)
    Share = Owners[Variable].replace({0: np.nan}).value_counts(normalize=True).sort_index() * 100
    Table1.append(pd.DataFrame({
        "question": get_question(Variable),
        "answer": Share.index.map(Labels),
        "percent": Share.round(1).values,
    }))
Table1 = pd.concat(Table1, ignore_index=True)
print(Table1)

In [ ]:
# Solution
for Variable in ["F27", "F28", "F29"]:
    Labels = get_labels(Variable)
    Share = Owners[Variable].replace({0: np.nan}).value_counts(normalize=True).sort_index() * 100
    plt.barh(Share.index.map(Labels), Share)
    plt.xlabel("Share of respondents (%)")
    plt.title(get_question(Variable))
    plt.show()

**3. Short answer (example):** the sample is **86% men**, **85% rural** and mostly **55 years or older**. Women and city dwellers are strongly under-represented. This partly reflects who owns forests (inheritance often went to sons, forests are in the countryside), and partly who answers surveys. Results describe *this sample*: to speak about "German forest owners" in general, researchers would need weights or a random sample.

## 3. Multiple choice questions (0/1 columns)

When several answers are possible, the survey stores **one column per option**, with `1` if the option was ticked and `0` otherwise. The prefix is shared: `F1_1_a`, `F1_1_b`, ... are all options of "How did you get your forest?".

A useful trick: **the mean of a 0/1 column is the share of people who ticked it.**

In [ ]:
AcquiredCols = ["F1_1_a", "F1_1_b", "F1_1_c", "F1_1_d", "F1_1_e"]
AcquiredShare = Owners[AcquiredCols].mean() * 100

# Use the codebook text (after the "?") as readable labels
AcquiredLabels = [get_question(Col).split("? ")[1] for Col in AcquiredCols]

plt.barh(AcquiredLabels, AcquiredShare)
plt.xlabel("Share of owners (%)")
plt.title(get_question("F1_1_a").split("?")[0] + "?")
plt.show()

> **Note:** the shares add up to more than 100%, because many owners inherited *and* bought forest. Never draw a pie chart of a multiple-choice question!

The same idea lets us compare two questions with the same options, for example **who takes care of the forest today** (`F12_*`) and **who will in 10 years** (`F13_*`). This is the generational question at the heart of the DIANA project.

In [ ]:
Options = ["a", "b", "c", "d", "e", "f"]
Care = pd.DataFrame({
    "today": [Owners[f"F12_{o}"].mean() * 100 for o in Options],
    "in 10 years": [Owners[f"F13_{o}"].mean() * 100 for o in Options],
}, index=[get_question(f"F12_{o}").split("? ")[1] for o in Options])
print(Care.round(1))

# A DataFrame can plot itself: one bar per column, side by side
Care.plot.barh()
plt.xlabel("Share of owners (%)")
plt.title("Who takes care of the forest?")
plt.show()

### Cross-tabulations

To see if an answer depends on **another** variable, use `pd.crosstab(rows, columns)`. With `normalize="index"`, each row sums to 1, so we compare groups of different sizes fairly.

Do owners who work professionally with forests (`F30`) ask experts for advice more often (`F17_b`)?

In [ ]:
Forestry = Owners["F30"].replace({0: np.nan}).map(get_labels("F30"))
print(pd.crosstab(Forestry, Owners["F17_b"], normalize="index").round(2))

## Exercise B

Owners have different ways to deal with their worries (drought, bark beetles, ...): `F17_a` to `F17_f`.

Tasks:
1. Compute the share of owners using each coping strategy, separately for owners who **work with forests** (`F30 == 1`) and those who **do not** (`F30 == 2`). Tip: `.groupby()` groups the rows by the values of a column before you compute the mean, e.g. `Owners.groupby("F30")[Cols].mean()`.
2. Plot both groups in one horizontal bar chart. Which strategies differ the most?

In [ ]:
# Solution
CopingCols = [f"F17_{o}" for o in "abcdef"]
Coping = Owners[Owners["F30"].isin([1, 2])].groupby("F30")[CopingCols].mean().T * 100
Coping.columns = ["works with forests", "does not"]
Coping.index = [get_question(Col).split("? ")[1] for Col in CopingCols]
print(Coping.round(1))

Coping.plot.barh()
plt.xlabel("Share of owners (%)")
plt.title("How do you deal with your worries?")
plt.show()

## 4. Likert and intensity scales

The block `F10` contains **44 statements** about the forest, each rated from `1 = strongly agree` to `5 = strongly disagree`. This is a **Likert scale**.

Likert answers are **ordinal**: *agree* is more than *neutral*, but we do not know if the step from *neutral* to *agree* is the same size as from *agree* to *strongly agree*. So:

- always look at the **distribution** (counts per category) first,
- the **median** is safe; the **mean** is common in papers but should be read with care.

In [ ]:
print(get_question("F10_v"))
print(Owners["F10_v"].value_counts().sort_index())

HeritageClean = Owners["F10_v"].replace({0: np.nan})
print("Median:", HeritageClean.median(), " Mean:", round(HeritageClean.mean(), 2))

### Reverse-coding

Here `1` means *strongly agree*, so a **low** number means **strong** agreement, which is confusing. We **reverse-code** the scale so that higher = more agreement: `new = 6 - old` (1 ↔ 5, 2 ↔ 4, 3 stays 3).

For the worries (`F15_*`, `1 = very strong` ... `5 = very weak`), the code `6 = does not apply` must be removed **before** reversing, otherwise it becomes `0`, a fake "extremely weak worry". `.where(condition)` keeps values where the condition is true and puts `NaN` elsewhere (same idea as in xarray, Class 5a).

In [ ]:
F10Cols = [Col for Col in Owners.columns if Col.startswith("F10_")]
Agreement = 6 - Owners[F10Cols].replace({0: np.nan})   # 5 = strongly agree

F15Cols = [f"F15_{o}" for o in "abcdefghijklmn"]      # F15_o ("other") left out
Worries = Owners[F15Cols].where((Owners[F15Cols] >= 1) & (Owners[F15Cols] <= 5))
Worries = 6 - Worries                                 # 5 = very strong worry

print(Agreement["F10_v"].value_counts().sort_index())

### Diverging stacked bar chart

The recommended plot for Likert data is the **diverging stacked bar chart** ([Heiberger & Robbins, 2014](https://doi.org/10.18637/jss.v057.i05)): each statement is one bar, disagreement goes to the **left** of zero, agreement to the **right**, and *neutral* is split in the middle. You see immediately which statements divide people.

Matplotlib has no ready-made function for it, but we can stack bars with the `left=` argument of `plt.barh`:

In [ ]:
def plot_diverging(Data, Labels, CategoryNames, Title):
    # Data: one column per item, values 1..5 (5 = most agreement / strongest)
    Categories = [1, 2, 3, 4, 5]
    Colors = ["#b2182b", "#ef8a62", "#d9d9d9", "#67a9cf", "#2166ac"]
    Percent = pd.DataFrame({c: (Data == c).sum() / Data.notna().sum() * 100 for c in Categories})

    Left = -(Percent[1] + Percent[2] + Percent[3] / 2)   # start of each bar
    for c, Color, Name in zip(Categories, Colors, CategoryNames):
        plt.barh(Labels, Percent[c], left=Left, color=Color, label=Name)
        Left = Left + Percent[c]

    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Share of respondents (%)")
    plt.title(Title)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()


Items = ["F10_k", "F10_y", "F10_3", "F10_v", "F10_s", "F10_4", "F10_12", "F10_z"]
ItemLabels = [get_question(Col)[:60] for Col in Items]
plot_diverging(
    Agreement[Items], ItemLabels,
    ["strongly disagree", "disagree", "neutral", "agree", "strongly agree"],
    "Relationship to the forest",
)

Almost everybody wants the forest to **stay a forest** and to be passed on, while **economic** statements split owners much more, and few see their forest as a burden.

## Exercise C

Tasks:
1. Rank the worries `F15_a` to `F15_n` from the strongest to the weakest (mean of the reversed scale `Worries`).
2. Draw the diverging chart of the worries, with the items in that order.
3. Which worries dominate: **climate-related risks** (drought, storms, bark beetles...) or **social and political** ones (politics, property rights, recognition...)? How many owners answered *does not apply* to "lack of own expertise" (`F15_i`)?

In [ ]:
# Solution
WorryOrder = Worries.mean().sort_values().index        # weakest first -> strongest at the top of the plot
print(Worries.mean().sort_values(ascending=False).round(2))

WorryLabels = [get_question(Col).split("because of ")[1].rstrip("?") for Col in WorryOrder]
plot_diverging(
    Worries[WorryOrder], WorryLabels,
    ["very weak", "weak", "medium", "strong", "very strong"],
    "How strongly do you worry about your forest because of...",
)

print("'Does not apply' for F15_i:", (Owners["F15_i"] == 6).sum())

**3. Short answer (example):** **drought** and **bark beetle** are by far the strongest worries, followed by storm damage: climate-related risks dominate. Politics and restrictions of property rights come next, clearly before lack of expertise or recognition. Note that a quarter of owners chose *does not apply* for some social items: removing code 6 changes the result a lot.

## 5. From single items to constructs

Social scientists rarely care about one statement alone. Several statements are designed to measure one underlying idea (a **construct**), such as *emotional attachment* or *economic orientation*. Before combining them, we check that they really go together.

### i. Correlation heatmap

For ordinal data we use the **Spearman** correlation (it works on ranks). `plt.imshow` draws a matrix as an image:

In [ ]:
Corr = Agreement.corr(method="spearman")

plt.figure(figsize=(9, 8))
plt.imshow(Corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(Corr)), Corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(Corr)), Corr.columns, fontsize=7)
plt.title("Correlations between the 44 F10 statements")
plt.show()

Blocks of blue squares are groups of statements that people answer in a similar way. Negative (red) cells often come from **negatively worded** items, e.g. `F10_12` "my forest is a burden".

### ii. Composite scores and Cronbach's alpha

A **composite score** is the mean of several items for each person (`.mean(axis=1)` averages across columns, i.e. per row). Its **reliability** is usually checked with **Cronbach's alpha**:

$$\alpha = \frac{k}{k-1}\left(1 - \frac{\sum_i \sigma^2_i}{\sigma^2_{total}}\right)$$

with $k$ the number of items, $\sigma^2_i$ the variance of each item and $\sigma^2_{total}$ the variance of the sum. As a rule of thumb, $\alpha > 0.7$ is acceptable. Alpha only says that the items move together; it does not prove that they measure a single idea (that is what factor analysis is for).

In [ ]:
def cronbach_alpha(Items):
    Items = Items.dropna()
    K = Items.shape[1]
    return K / (K - 1) * (1 - Items.var().sum() / Items.sum(axis=1).var())


FutureItems = ["F10_w", "F10_x", "F10_y"]     # adapt to climate change, fit for the future, next generations
for Col in FutureItems:
    print(Col, get_question(Col))

Owners["future_score"] = Agreement[FutureItems].mean(axis=1)
print("Cronbach's alpha:", round(cronbach_alpha(Agreement[FutureItems]), 2))

### iii. Factor analysis: letting the data find the constructs

With 44 items, choosing groups by hand is hard. **Factor analysis** assumes that a few hidden factors explain the correlations between items. Each item gets a **loading** on each factor (similar to a correlation): items with high loadings on the same factor form a construct. A **varimax rotation** makes the loadings easier to read (each item loads mainly on one factor).

We use scikit-learn. Like most scikit-learn tools it works in two steps: create the model, then `.fit()` it to the data. Items are standardised first (mean 0, standard deviation 1), and rows with missing answers are dropped.

In [ ]:
from sklearn.decomposition import FactorAnalysis

Complete = Agreement.dropna()
Standardised = (Complete - Complete.mean()) / Complete.std()
print("Respondents used:", len(Complete))

Fa = FactorAnalysis(n_components=4, rotation="varimax", random_state=0)
Fa.fit(Standardised)

Loadings = pd.DataFrame(Fa.components_.T, index=Complete.columns,
                        columns=["factor 1", "factor 2", "factor 3", "factor 4"])

for Factor in Loadings.columns:
    Top = Loadings[Factor].abs().sort_values(ascending=False).index[:6]
    print(f"\n{Factor}:")
    for Col in Top:
        print(f"  {Loadings.loc[Col, Factor]:+.2f}  {get_question(Col)[:80]}")

Reading the top items, we can name the factors, for example:
- **engagement / enjoyment**: I like working in my forest, time spent meaningfully, place of recreation (and *not* a burden),
- **economic orientation**: economic benefit, cover the costs, financial security, new income,
- **future / adaptation**: adapt to climate change, fit for the future, next generations,
- **ecological values**: habitat for plants and animals, water balance, carbon storage.

The **sign** of a factor is arbitrary: what matters is which items load strongly together, and whether some load in the opposite direction (e.g. "burden" against "I like working in my forest"). This is how studies of forest owners' values and objectives build their scales. The number of factors is a choice: try 3 or 5 and see if the factors stay interpretable.

## Exercise D

Tasks:
1. Build two composite scores (on the reversed `Agreement` scale):
   - `emotional_score` from `F10_g`, `F10_3`, `F10_5`, `F10_11`, `F10_1` (identity, emotional connection, memories, awe, beauty),
   - `economic_score` from `F10_s`, `F10_t`, `F10_4`, `F10_2`, `F10_n`.
2. Compute Cronbach's alpha of both scales. Are they reliable?
3. Plot the two distributions in one histogram (`plt.hist(..., alpha=0.5)`). What is more important to these owners?

In [ ]:
# Solution
EmotionalItems = ["F10_g", "F10_3", "F10_5", "F10_11", "F10_1"]
EconomicItems = ["F10_s", "F10_t", "F10_4", "F10_2", "F10_n"]

Owners["emotional_score"] = Agreement[EmotionalItems].mean(axis=1)
Owners["economic_score"] = Agreement[EconomicItems].mean(axis=1)

print("Alpha emotional:", round(cronbach_alpha(Agreement[EmotionalItems]), 2))
print("Alpha economic: ", round(cronbach_alpha(Agreement[EconomicItems]), 2))
print(Owners[["emotional_score", "economic_score"]].describe().round(2))

In [ ]:
# Solution
Bins = np.arange(1, 5.25, 0.25)
plt.hist(Owners["emotional_score"], bins=Bins, alpha=0.5, label="emotional")
plt.hist(Owners["economic_score"], bins=Bins, alpha=0.5, label="economic")
plt.xlabel("Score (1 = strongly disagree, 5 = strongly agree)")
plt.ylabel("Number of owners")
plt.legend()
plt.show()

**Short answer (example):** both scales are reliable ($\alpha \approx 0.8$). Emotional attachment is higher and less spread than economic orientation: most owners are emotionally attached, while the economic role of the forest divides them.

## 6. Comparing groups and testing

### i. `groupby`

`groupby` splits the table by the values of a column, then applies a calculation to each group. Do owners who **inherited** their forest (`F1_1_a == 1`) care more about honouring the heritage of previous generations (`F10_v`)?

In [ ]:
Owners["inherited"] = Owners["F1_1_a"]
Owners["heritage_agree"] = Agreement["F10_v"]

print(Owners.groupby("inherited")["heritage_agree"].agg(["count", "median", "mean"]).round(2))

# Share of owners who agree or strongly agree (4 or 5)
print((Owners["heritage_agree"] >= 4).groupby(Owners["inherited"]).mean().round(2))

### ii. Is the difference "real"? Mann-Whitney U test

With a sample, a difference can appear by chance. A **statistical test** estimates how likely such a difference would be if the two groups were in truth identical (the **p-value**). For ordinal data comparing two groups, the standard test is **Mann-Whitney U** (`scipy.stats.mannwhitneyu`).

A p-value only tells you *whether* there is a difference, not *how big* it is. Always report an **effect size**. For Mann-Whitney, the **rank-biserial correlation** $r = 1 - \frac{2U}{n_1 n_2}$ goes from -1 to 1 (about 0.1 small, 0.3 medium, 0.5 large).

In [ ]:
from scipy import stats

Inherited = Owners.loc[Owners["inherited"] == 1, "heritage_agree"].dropna()
NotInherited = Owners.loc[Owners["inherited"] == 0, "heritage_agree"].dropna()

Result = stats.mannwhitneyu(NotInherited, Inherited)
EffectSize = 1 - 2 * Result.statistic / (len(Inherited) * len(NotInherited))
print(f"U = {Result.statistic:.0f}, p = {Result.pvalue:.2g}, rank-biserial r = {EffectSize:.2f}")

### iii. Relationship between two ordinal variables: Spearman

Are owners who feel emotionally attached also the ones who visit their forest more often (`F34`)? Careful with the coding: in `F34`, `1 = daily` and `6 = less often`, so a **negative** correlation means *more attached = more frequent visits*.

In [ ]:
Visits = Owners["F34"].replace({0: np.nan})
print(get_labels("F34"))

Rho, P = stats.spearmanr(Owners["emotional_score"], Visits, nan_policy="omit")
print(f"Spearman rho = {Rho:.2f}, p = {P:.2g}")

### iv. Two categorical variables: chi-square test

People heard about the survey through different channels. Are owners reached via a **forestry organisation** (`F40_f`) more often in contact with a **forest owner association** (`F14_a`)? For two categorical variables, we use the **chi-square test** on the cross-table of counts.

In [ ]:
Counts = pd.crosstab(Owners["F40_f"], Owners["F14_a"])
print(Counts)
print(pd.crosstab(Owners["F40_f"], Owners["F14_a"], normalize="index").round(2))

Chi2, P, Dof, Expected = stats.chi2_contingency(Counts)
print(f"chi2 = {Chi2:.1f}, p = {P:.2g}")

This is a classic **sampling bias**: owners contacted through forestry networks are, not surprisingly, better networked. It is exactly why DIANA also recruited through non-forestry channels.

> **Caveats about tests**
> - A small p-value is not a large or important effect; look at effect sizes and plots.
> - If you run 44 tests, about 2 will be "significant" at p < 0.05 by pure chance (**multiple testing**). Correct (e.g. Bonferroni: divide 0.05 by the number of tests) or decide your hypotheses in advance.
> - Tests assume a random sample. Ours is not, so p-values are indications, not proofs.

### v. Regression with statsmodels

Tests compare one variable at a time. A **regression** estimates the effect of several variables **at once**, each one "controlling" for the others. `statsmodels` lets you write the model as a formula, like in R: `outcome ~ x1 + x2`.

In [ ]:
import statsmodels.formula.api as smf

Owners["works_in_forestry"] = (Owners["F30"] == 1).astype(int)
Owners["age_class"] = Owners["F26"].replace({0: np.nan})

Model = smf.ols("emotional_score ~ inherited + works_in_forestry + age_class", data=Owners).fit()
print(Model.summary().tables[1])
print("R2 =", round(Model.rsquared, 3))

Each coefficient is the change in the score when that variable increases by one, all others being equal. The table also gives a p-value and a 95% confidence interval for each coefficient.

An ordinary regression treats the outcome as a continuous number. For a **single Likert item**, the correct model is an **ordinal logistic regression** (`OrderedModel`), which only uses the order of the categories:

In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

Data = Owners[["heritage_agree", "inherited", "works_in_forestry", "age_class"]].dropna()
Ordinal = OrderedModel(Data["heritage_agree"].astype(int),
                       Data[["inherited", "works_in_forestry", "age_class"]],
                       distr="logit").fit(method="bfgs", disp=False)
print(Ordinal.summary().tables[1])

A positive coefficient means higher odds of a higher category (more agreement). `np.exp(coefficient)` gives the **odds ratio**: here, inheriting the forest multiplies the odds of agreeing more with "honour the heritage" by about 3.

## Exercise E

Does the **size of the forest** explain the economic orientation of owners? The problem: the size (`F31_b`) was typed freely by respondents, so it is **text**: `"1,5"`, `"approx. 8 ha"`, `"less than 1 ha"`, `"12 hectares"`...

Tasks:
1. Look at the different formats with `value_counts()`.
2. Convert `F31_b` into a number of hectares in a new column `hectares`:
   - `-99` and texts without a number become `NaN`,
   - "less than 1 ha" (and similar) becomes `0.5`,
   - otherwise, extract the first number with `.str.extract(r"(\d+(?:[.,]\d+)?)")` (a **regular expression**, look it up!) and replace the decimal comma by a point before `.astype(float)`.
3. Forest sizes are very skewed (a few owners have hundreds of hectares). Create `log_hectares = np.log1p(hectares)` (`log1p(x) = log(1 + x)` also works for 0).
4. Fit `economic_score ~ log_hectares + works_in_forestry + age_class + inherited`. Which variables matter? Interpret the coefficient of `log_hectares`.

In [ ]:
# Solution
print(Owners["F31_b"].value_counts().head(20))

Size = Owners["F31_b"].str.lower()
Number = Size.str.extract(r"(\d+(?:[.,]\d+)?)")[0].str.replace(",", ".").astype(float)
LessThanOne = Size.str.contains(r"less than|under one|< ?1", regex=True)

Owners["hectares"] = Number
Owners.loc[LessThanOne, "hectares"] = 0.5
Owners.loc[Owners["F31_b"].isin(["-99", "-66"]), "hectares"] = np.nan

print(Owners["hectares"].describe().round(1))

In [ ]:
# Solution
Owners["log_hectares"] = np.log1p(Owners["hectares"])

plt.hist(Owners["log_hectares"].dropna(), bins=30)
plt.xlabel("log(1 + hectares)")
plt.ylabel("Number of owners")
plt.show()

EconomicModel = smf.ols(
    "economic_score ~ log_hectares + works_in_forestry + age_class + inherited", data=Owners
).fit()
print(EconomicModel.summary().tables[1])
print("R2 =", round(EconomicModel.rsquared, 3), " n =", int(EconomicModel.nobs))

**4. Short answer (example):** owners of **larger forests** and owners who **work in forestry** are clearly more economically oriented; older owners slightly less. Inheritance does not matter once the other variables are included. Since size is on a log scale, doubling the forest area raises the economic score by about $0.24 \times \log(2) \approx 0.17$ points. Together these variables explain about a quarter of the differences between owners ($R^2 \approx 0.25$).

## 7. Text as data: machine learning on open answers

Open answers are rich but hard to count. The traditional method is **qualitative coding**: researchers read every answer and assign categories, ideally two coders independently, then check their agreement (**inter-coder reliability**). This is slow for hundreds of answers.

Machine learning can help in two ways (see Grimmer & Stewart, 2013, *Text as Data*):
- **unsupervised**: let an algorithm find recurring **topics**, then read examples to interpret them,
- **supervised**: code a **sample** by hand, train a model on it, and let it code the rest.

In both cases, the computer never replaces reading: you **validate** the results by going back to the answers.

> The answers are in **German**, the language of the survey. The methods do not depend on the language, only the list of *stop words* (very frequent words like "und", "die") does.

### i. Bag of words

Models cannot read text directly: we turn each answer (a **document**) into a row of word counts, the **document-term matrix**. Word order is lost, hence the name *bag of words*. In scikit-learn this is `CountVectorizer`.

Our first documents are the **free associations** (`F4_a` to `F4_e`): each owner wrote up to five words that come to mind about their forest. We join them into one document per owner, linking multi-word expressions with `_` (e.g. `viel_arbeit`) so that they stay one token.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

KeywordCols = ["F4_a", "F4_b", "F4_c", "F4_d", "F4_e"]
Keywords = Owners[KeywordCols].replace({"-99": "", "-66": ""}).fillna("")
KeywordDocs = Keywords.apply(lambda Row: " ".join(w.replace(" ", "_") for w in Row if w), axis=1)
print(KeywordDocs.head())

# token_pattern: a token is anything between spaces
Counter = CountVectorizer(token_pattern=r"[^ ]+")
Dtm = Counter.fit_transform(KeywordDocs)
print("Document-term matrix:", Dtm.shape)   # owners x distinct words

WordCounts = pd.Series(np.asarray(Dtm.sum(axis=0)).ravel(), index=Counter.get_feature_names_out())
Top20 = WordCounts.sort_values().tail(20)

plt.barh(Top20.index, Top20)
plt.xlabel("Number of owners mentioning the word")
plt.title("Spontaneous associations with 'my forest'")
plt.show()

*Arbeit* (work), *Erholung* (recreation), *Natur* (nature), *Borkenkäfer* (bark beetle) and *Brennholz* (firewood) lead: the forest is at the same time a workplace, a place to relax, nature, a threat and a resource.

### ii. Topic modelling with TF-IDF and NMF

Counting words does not tell us which words **go together**. A **topic model** finds groups of words that often appear in the same documents.

1. **TF-IDF** (`TfidfVectorizer`) re-weights the counts: words used by everybody get a low weight, words typical of some documents a high weight. `min_df=3` ignores words used by fewer than 3 owners.
2. **NMF** (non-negative matrix factorisation) decomposes the matrix into $k$ topics. Each topic is a list of word weights; each owner gets a weight on each topic.

The number of topics $k$ is **your choice**: try several and keep the most interpretable one.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

Tfidf = TfidfVectorizer(token_pattern=r"[^ ]+", min_df=3)
KeywordTfidf = Tfidf.fit_transform(KeywordDocs)
Words = Tfidf.get_feature_names_out()

KeywordNmf = NMF(n_components=4, init="nndsvda", random_state=0, max_iter=500)
KeywordTopics = KeywordNmf.fit_transform(KeywordTfidf)   # owners x topics

for i, Weights in enumerate(KeywordNmf.components_):
    TopWords = [Words[j] for j in Weights.argsort()[::-1][:8]]
    print(f"Topic {i}: {', '.join(TopWords)}")

The topics can be named, for example: **work and wood** (Arbeit, Brennholz, Hobby), **nature** (Natur, Tiere, Ruhe), **threats and forest conversion** (Borkenkäfer, Trockenheit, Klimawandel, Waldumbau) and **sustainability and recreation** (Nachhaltigkeit, Erholung, Naturschutz).

The topic weights are new **numeric variables** for each owner, so we can connect the text to the rest of the survey. Do owners whose associations are about **threats** also report stronger worries about drought, storm and bark beetle?

In [ ]:
ThreatTopic = 2   # check the topic number in the output above
ClimateWorry = Worries[["F15_a", "F15_b", "F15_e"]].mean(axis=1)

Rho, P = stats.spearmanr(KeywordTopics[:, ThreatTopic], ClimateWorry, nan_policy="omit")
print(f"Spearman rho = {Rho:.2f}, p = {P:.2g}")

### iii. Topics in open answers (disaster scenario)

`F19` asked: *"What would you do if a disaster killed half of the trees in your forest and the other half were threatened?"*. These are full sentences, so we remove **stop words** (words that carry no topic). scikit-learn has a built-in English list (`stop_words="english"`); for German we give our own list.

In [ ]:
print(get_question("F19"))

StopWords = (
    "der die das den dem des ein eine einen einem einer und oder aber ich mich mir mein meine meinen meinem "
    "wir uns es sie er man zu zum zur in im an am auf aus bei mit nach von vor für um über unter durch "
    "ist sind war wird werden würde würden hat haben habe sein kann könnte muss müsste soll so als auch dann "
    "wenn was wie noch mehr nicht nur schon sehr ggf bzw evtl da dass ob alle alles"
).split()

Disaster = Owners.loc[~Owners["F19"].isin(["-99", "-66"]), "F19"]
print("Answers:", len(Disaster))

DisasterTfidf = TfidfVectorizer(stop_words=StopWords, min_df=5)
DisasterMatrix = DisasterTfidf.fit_transform(Disaster)
DisasterWords = DisasterTfidf.get_feature_names_out()

DisasterNmf = NMF(n_components=4, init="nndsvda", random_state=0, max_iter=500)
DisasterTopics = DisasterNmf.fit_transform(DisasterMatrix)

for i, Weights in enumerate(DisasterNmf.components_):
    print(f"\nTopic {i}: {', '.join(DisasterWords[j] for j in Weights.argsort()[::-1][:8])}")
    # Validation: read the 3 answers most typical of this topic
    for Row in DisasterTopics[:, i].argsort()[::-1][:3]:
        print("   -", Disaster.iloc[Row][:120])

Reading the typical answers is the **validation** step: do the answers really share an idea? Here we find salvaging and selling the wood (*aufarbeiten, verkaufen, Brennholz*), clearing and replanting (*abholzen, neu anpflanzen*), and reforesting with suitable, climate-adapted species (*Baumarten, Wiederbewaldung*). If a topic mixes unrelated answers, change $k$, `min_df` or the stop words.

### iv. Supervised classification: learning from a hand-coded sample

`F22` asked who would **decide** what happens in the disaster scenario. We (the instructors) read and coded 150 answers into four categories: `self`, `family`, `forester_or_expert` and `other`. They are in `diana_text_labels.csv`. We train a model on part of them and **test** it on the rest, which the model has never seen.

The model is a **logistic regression** on TF-IDF features (`ngram_range=(1, 2)` also uses pairs of words, like "meine familie"). `class_weight="balanced"` gives more weight to rare categories.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

TextLabels = pd.read_csv("../processed_data/diana_text_labels.csv")
Coded = TextLabels[TextLabels["question"] == "F22"].merge(Owners[["respondent_id", "F22"]], on="respondent_id")
print(Coded["label"].value_counts())

TextTrain, TextTest, LabelTrain, LabelTest = train_test_split(
    Coded["F22"], Coded["label"], test_size=0.3, random_state=0, stratify=Coded["label"]
)

Vectorizer = TfidfVectorizer(ngram_range=(1, 2))
Classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
Classifier.fit(Vectorizer.fit_transform(TextTrain), LabelTrain)

Predicted = Classifier.predict(Vectorizer.transform(TextTest))
print(classification_report(LabelTest, Predicted, zero_division=0))

ConfusionMatrixDisplay.from_predictions(LabelTest, Predicted, xticks_rotation=45)
plt.show()

How to read the report: **precision** = among answers predicted as *family*, how many really are; **recall** = among real *family* answers, how many were found. The model works well for `self` and `family`, less for rare categories: with only a handful of examples it cannot learn them. That is a general lesson: rare categories need more hand-coded examples.

Once the model is good enough, we let it code **all** answers and use the result as a new variable:

In [ ]:
Answered = ~Owners["F22"].isin(["-99", "-66"])
Owners.loc[Answered, "decides_disaster"] = Classifier.predict(Vectorizer.transform(Owners.loc[Answered, "F22"]))

print(Owners["decides_disaster"].value_counts())
print(pd.crosstab(Owners["inherited"], Owners["decides_disaster"], normalize="index").round(2))

> **Discussion: can we trust machine coding?**
> - The model reproduces **our** coding choices, including our mistakes and biases. The hand-coded sample must be good (ideally two coders, with their agreement measured, e.g. Cohen's kappa).
> - Always report the performance on a **test set**, never on the training data.
> - Bag-of-words ignores meaning and word order: "nicht ich" (*not me*) looks like "ich" (*me*). Modern **embeddings** and **large language models** capture meaning much better; we will see LLMs in Classes 7 and 8. The same validation rules apply to them.

## Exercise F

`F25` asked who would decide if the forest were **sold**.

Tasks:
1. Build the same supervised pipeline for `F25` with the labels of `diana_text_labels.csv` (`question == "F25"`). Look at the label counts first: a category with only one or two examples cannot be learned (and `stratify` fails). Merge rare categories into `other` before splitting.
2. Evaluate the model on a test set.
3. Predict the category of all `F25` answers and compare with the disaster scenario (`decides_disaster`) using `pd.crosstab`. Do the same people decide in both situations?

In [ ]:
# Solution
CodedSale = TextLabels[TextLabels["question"] == "F25"].merge(Owners[["respondent_id", "F25"]], on="respondent_id")
print(CodedSale["label"].value_counts())

CodedSale["label"] = CodedSale["label"].replace({"forester_or_expert": "other"})

SaleTrain, SaleTest, SaleLabelTrain, SaleLabelTest = train_test_split(
    CodedSale["F25"], CodedSale["label"], test_size=0.3, random_state=0, stratify=CodedSale["label"]
)
SaleVectorizer = TfidfVectorizer(ngram_range=(1, 2))
SaleClassifier = LogisticRegression(max_iter=1000, class_weight="balanced")
SaleClassifier.fit(SaleVectorizer.fit_transform(SaleTrain), SaleLabelTrain)

print(classification_report(SaleLabelTest, SaleClassifier.predict(SaleVectorizer.transform(SaleTest)), zero_division=0))

In [ ]:
# Solution
AnsweredSale = ~Owners["F25"].isin(["-99", "-66"])
Owners.loc[AnsweredSale, "decides_sale"] = SaleClassifier.predict(
    SaleVectorizer.transform(Owners.loc[AnsweredSale, "F25"])
)
print(pd.crosstab(Owners["decides_disaster"], Owners["decides_sale"]))

**3. Short answer (example):** most owners decide **alone** in both scenarios, and owners who involve their **family** in one situation usually do so in the other. Experts appear in the disaster scenario (a technical question) but almost never for a sale (a family and property question).

## Exercise G (Bonus): a typology of forest owners

A classic result of forest-owner research is a **typology**: groups of owners with similar objectives, used to design advice and policy (e.g. [Tiebel et al., 2024](https://doi.org/10.1002/pan3.10579) found *multiple-use*, *conventional* and *conservation-oriented* owners in Lower Saxony). They are built with **cluster analysis**.

Tasks:
1. Build (or reuse) five composite scores from the factor analysis: `emotional_score`, `economic_score`, `future_score`, plus an **engagement** score (`F10_b`, `F10_8`, `F10_i`, `F10_5`) and an **ecological** score (`F10_l`, `F10_r`, `F10_m`, `F10_0`, `F10_k`).
2. Standardise them (`sklearn.preprocessing.StandardScaler`) and draw a **dendrogram** with `scipy.cluster.hierarchy.linkage(..., method="ward")` and `dendrogram(...)`. How many clusters would you choose?
3. Run `AgglomerativeClustering(n_clusters=k, linkage="ward")` and describe each cluster with `groupby().mean()` of the scores. Add the share of owners who work in forestry and the mean weight of each keyword topic.
4. Check with a **Kruskal-Wallis** test (`stats.kruskal`, the extension of Mann-Whitney to more than two groups) that the clusters differ in their economic score.
5. Give each type a name.

In [ ]:
# Solution
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram

Owners["engagement_score"] = Agreement[["F10_b", "F10_8", "F10_i", "F10_5"]].mean(axis=1)
Owners["ecological_score"] = Agreement[["F10_l", "F10_r", "F10_m", "F10_0", "F10_k"]].mean(axis=1)
ScoreCols = ["emotional_score", "economic_score", "future_score", "engagement_score", "ecological_score"]

Scores = Owners[ScoreCols].dropna()
ScoresStd = StandardScaler().fit_transform(Scores)

plt.figure(figsize=(10, 4))
dendrogram(linkage(ScoresStd, method="ward"), no_labels=True)
plt.ylabel("Ward distance")
plt.title("Dendrogram of forest owners")
plt.show()

In [ ]:
# Solution
Clusters = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(ScoresStd)
Owners.loc[Scores.index, "owner_type"] = Clusters

Topics = pd.DataFrame(KeywordTopics, columns=[f"topic_{i}" for i in range(KeywordTopics.shape[1])])
Profile = pd.concat([Owners[ScoreCols + ["works_in_forestry", "owner_type"]], Topics], axis=1)
print(Owners["owner_type"].value_counts())
print(Profile.groupby("owner_type").mean().round(2).T)

Groups = [Owners.loc[Owners["owner_type"] == k, "economic_score"] for k in range(3)]
print(stats.kruskal(*Groups))

**5. Short answer (example):** with 3 clusters we find
- **multifunctional owners**: high on every dimension, including economic use,
- **attached, non-economic owners**: strong emotional and ecological values, low economic orientation,
- **distant owners**: lower scores everywhere, the forest plays a smaller role in their life.

This resembles published typologies, but the number and names of clusters are **interpretations**: another $k$, other scores or another method can give other types. Researchers justify their choice with the dendrogram, with stability checks, and with what makes sense for the research question.